In [ ]:
import sys, os

# 1つ上のディレクトリにパスを通す
sys.path.append(os.path.abspath('../'))

import pandas as pd

from module.Feature import UserFeature, ItemFeature

In [ ]:
train = pd.read_csv('../data/train/train_A.tsv', sep='\t')


In [ ]:
train_user = UserFeature(train)
train_product = ItemFeature(train)

train_user.event_count()
train_product.event_count()


In [ ]:
time_df = pd.DataFrame({
    'start_time': ['06:00', '10:00', '15:00', '18:00', '00:00'],
    'end_time': ['10:00', '15:00', '18:00', '23:59', '06:00'],
    'time_zone': ['morning', 'daytime', 'evening', 'night', 'latenight']
})


In [ ]:
for _, row in time_df.iterrows():
    st = row['start_time']
    ed = row['end_time']
    tz = row['time_zone']

    train_user.time_count(st, ed, tz)
    train_product.time_count(st, ed, tz)


In [ ]:
train_user.weekday_count()
train_product.weekday_count()

train_user.save('product_id', 'user_id', 'user')

train_product.save('user_id', 'product_id', 'product')


In [ ]:
# relevance の取得のために user_df, product_df から必要な列だけに絞る
user_relevance = train_user.df[["user_id", "relevance"]].rename(columns={"relevance": "user_relevance"})
product_relevance = train_product.df[["product_id", "relevance"]].rename(columns={"relevance": "product_relevance"})


In [ ]:
# log_df にユーザーと商品から relevance を join（マージ）
interaction_df = train[["user_id", "product_id"]].drop_duplicates()  # 重複クリックを除去（必要に応じて）
interaction_df = interaction_df.merge(user_relevance, on="user_id", how="left")
interaction_df = interaction_df.merge(product_relevance, on="product_id", how="left")


In [ ]:
interaction_df.to_csv('/home/whitesalena/python/pytorch/programs/Competition/project_signate/opt_2/data/interaction_A.tsv', sep='\t', index=False)